# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omar320250068-prog/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This notebook does two things in the same spirit. First it reads two findings from FlyRank's own
research paper the way a careful reviewer would — respectfully and concretely, asking *where does
the label come from?* and *does the validation design carry the claim?* Then it turns the same
lens on **my own Week-5 model**: an honest split, a leakage hunt, real failure examples, and a
rewrite of my boldest claim into safe language.

Rule I hold myself to throughout: **observed, measured, directional, decision-support** — never
causal, never "proven".

The two halves share one idea: a claim is only as strong as the *design* standing behind it. The
paper's headline findings already disclose theirs; my model had not, so I disclose mine here.

## 1. Two paper findings + my methodology questions

I picked the two findings from the FlyRank paper that overlap most with my own lane, because those
are the ones I most need to understand honestly.

### Finding #4 — The Freshness Multiplier ("3.2x health, 57x impressions from refresh")

**The claim as published:** *"365+ day content that was refreshed within 30 days shows a 3.2x health
boost (from 10.7 to 34.5) and 57x more impressions (from 71 to 4039)."* The paper is careful to say
the `361+` ratio bucket is unstable and that refresh timing is *"one of the strongest measured
levers."*

**My methodology question — where does the comparison come from?** The 3.2x / 57x numbers compare
pages *that were refreshed* against pages *that were not*. But how did a page end up refreshed?
Editorial teams typically refresh pages that already have proven demand or strategic value — so the
refreshed group may be **selected for strength before the measurement even starts**. That is a
selection / confounding effect, not necessarily a treatment effect of refreshing.

> *Constructive framing:* the paper already calls this an observational study and declines to claim
> causation, which is exactly right. The concrete question I would ask is: **is there a matched
> baseline?** If refreshed and untouched mature pages were compared *after matching on
> pre-refresh impressions* (or if the same pages were compared to their own prior-30-day window),
> the 57x would be far more believable as a refresh signal rather than a selection signal. Without
> that, the honest reading is *"refreshed pages in this portfolio grew more"* — directional, not a
> guaranteed lever.

### Finding #3 — Click Capture by Position Tier ("88% drop from top 3 to deep")

**The claim as published:** *"Weighted portfolio CTR declines sharply as visibility moves away from
the top ... 0.423% in top 3 ... 88% drop from top 3 to deep."* The paper notes these are
**portfolio-level weighted CTRs** (total clicks ÷ total impressions per tier), explicitly not per-row
averages, and fixes an earlier bug that produced impossible values above 100%.

**My methodology question — does the aggregate design carry the claim?** The move to *weighted* CTR
is a genuine improvement: it removes the per-row averaging artefact, and the paper says so. My
question is about **what else moves with position**. Position tiers are not a clean
rank-only variable: a page sitting at position 30 is often a *different kind of query* — lower
volume, more long-tail, different intent — than a page at position 2. So the "88% drop" may bundle
**rank effect + query-mix effect**. The design cannot separate them, and to its credit the paper
never claims to.

> *Constructive framing:* the paper's own interpretation rule ("external SEO beliefs are treated as
> hypotheses") applies here too. The concrete question I would ask is: **could the tier gap be shown
> at the same query intent / volume band?** If CTR still falls steeply *within* a single intent-and-
> volume band as position worsens, the rank story strengthens; if the gap mostly closes, the drop was
> partly query mix. Either result is useful — the point is to *measure* it rather than assume it.

### What I take from both

Both findings are honest about being observational, and both would be reviewed better with **one
matched/controlled comparison** added. That is the standard I now apply to my own model below.

## 2. My model under an honest split (before/after)

The card asks for a **before/after of at least one honest-split improvement**. My Week-5 notebook
used a **client-holdout** split already — so here I make the gap explicit by running the *same
features and the same model three ways* and printing the difference:

| Split | What it simulates | Honesty |
|---|---|---|
| **Random 4-fold** (stratified) | pages mixed freely across folds | ✗ too easy — same client in train and test |
| **Grouped-client 4-fold** (`GroupKFold` on `client_id`) | clients never seen in training | ✓ the honest split for this lane |

The **gap between them is itself the finding**: it measures how much the random split was letting
the model *memorise the client* rather than learn the world. I report out-of-fold metrics (never
in-sample), next to the base rate.

## 0. Setup

Repo-root resolution so the notebook runs locally and on Colab (which clones the repo).

In [1]:
import os, sys, subprocess, json
import numpy as np, pandas as pd

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/omar320250068-prog/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"
if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    if os.path.basename(os.getcwd()) == "notebooks":
        os.chdir("..")
    if not os.path.isdir("data/raw"):
        for cand in (r"C:\Users\OMAR HINDAWY\flyrank-ml-internship", os.getcwd()):
            if os.path.isdir(os.path.join(cand, "data", "raw")):
                os.chdir(cand); break

import sklearn
from sklearn.model_selection import StratifiedKFold, GroupKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

os.makedirs("work/outputs", exist_ok=True)
SEED = 42
print("Working dir:", os.getcwd())
print("scikit-learn:", sklearn.__version__, "| pandas:", pd.__version__, "| numpy:", np.__version__)

Working dir: C:\Users\OMAR HINDAWY\flyrank-ml-internship
scikit-learn: 1.9.1 | pandas: 3.0.6 | numpy: 2.5.4


In [2]:
# Rebuild the Week-5 lane slice + proxy label + LEAKAGE-SAFE feature matrix (identical to w05).
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
vis = df[(df["impressions_90d"] >= 500) & (df["avg_position"] > 0) &
         (df["avg_position"] <= 20)].copy()
vis["pos_tier"] = pd.cut(vis["avg_position"], bins=[0, 3, 10, 20],
                         labels=["top_3", "page_1", "striking"]).astype(str)
tier_median = vis.groupby("pos_tier")["ctr"].transform("median")
vis["ctr_vs_tier"] = vis["ctr"] / tier_median.replace(0, np.nan)
vis["under_capturing"] = ((vis["ctr_vs_tier"] < 0.50) &
                           (vis["impressions_90d"] >= 1000)).astype(int)

# Same honest feature set as Week 5: EXCLUDED clicks_90d & ctr (they define the label),
# ctr_vs_tier (label source), trend_*/is_declining_label (label trap).
num_feats = ["impressions_90d", "avg_position", "content_age_days",
             "days_since_last_update", "sessions_90d", "engagement_rate",
             "scroll_rate", "word_count", "search_volume"]
cat_feats = ["pos_tier", "main_intent", "content_type", "position_tier",
             "age_tier", "freshness_tier"]
num_feats = [c for c in num_feats if c in vis.columns]
cat_feats = [c for c in cat_feats if c in vis.columns]

X = vis[num_feats + cat_feats].copy()
for c in num_feats:
    X[c] = pd.to_numeric(X[c], errors="coerce")
X = pd.concat([X[num_feats].fillna(0),
               pd.get_dummies(X[cat_feats].astype(str))], axis=1)
y = vis["under_capturing"].values
groups = vis["client_id"].values
print(f"Slice: {len(vis):,} pages | clients: {vis['client_id'].nunique()} | "
      f"features: {X.shape[1]} | base rate: {y.mean():.3f}")

Slice: 12,023 pages | clients: 28 | features: 31 | base rate: 0.193


In [3]:
# Out-of-fold predictions under each split. Same model both times — only the split changes.
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return float(np.asarray(labels)[order[:k]].mean())

def oof_predictions(splitter, use_groups):
    preds = np.zeros(len(y))
    folds = splitter.split(X, y, groups) if use_groups else splitter.split(X, y)
    for tr, te in folds:
        model = RandomForestClassifier(n_estimators=300, max_depth=8,
                                       class_weight="balanced",
                                       random_state=SEED, n_jobs=-1)
        model.fit(X.iloc[tr], y[tr])
        preds[te] = model.predict_proba(X.iloc[te])[:, 1]
    return preds

random_preds = oof_predictions(StratifiedKFold(n_splits=4, shuffle=True, random_state=SEED), False)
grouped_preds = oof_predictions(GroupKFold(n_splits=4), True)

def score_row(name, p):
    return {"split": name, "P@20": precision_at_k(p, y, 20),
            "P@50": precision_at_k(p, y, 50),
            "ROC_AUC": roc_auc_score(y, p), "AP": average_precision_score(y, p)}

results = pd.DataFrame([score_row("random (4-fold, too easy)", random_preds),
                        score_row("grouped-client (4-fold, honest)", grouped_preds)])
results.loc[len(results)] = {"split": "base_rate", "P@20": y.mean(), "P@50": y.mean(),
                             "ROC_AUC": 0.5, "AP": y.mean()}
print(results.to_string(index=False))

gap = {"P@20": results.loc[0, "P@20"] - results.loc[1, "P@20"],
       "ROC_AUC": results.loc[0, "ROC_AUC"] - results.loc[1, "ROC_AUC"],
       "AP": results.loc[0, "AP"] - results.loc[1, "AP"]}
print("\nGAP (random - grouped) = memorisation effect:",
      {k: round(v, 4) for k, v in gap.items()})

                          split    P@20    P@50  ROC_AUC       AP
      random (4-fold, too easy) 0.90000 0.86000 0.841366 0.538999
grouped-client (4-fold, honest) 0.45000 0.58000 0.780454 0.430377
                      base_rate 0.19288 0.19288 0.500000 0.192880

GAP (random - grouped) = memorisation effect: {'P@20': np.float64(0.45), 'ROC_AUC': np.float64(0.0609), 'AP': np.float64(0.1086)}


In [4]:
# Receipt: save the before/after comparison so the numbers can't drift from the claims.
receipt = {
    "notebook": "w06_validation_audit",
    "lane": "CTR / Engagement Opportunity Scoring",
    "label": "under_capturing = (ctr_vs_tier < 0.50) & (impressions_90d >= 1000)",
    "slice_rows": int(len(vis)),
    "n_clients": int(vis["client_id"].nunique()),
    "base_rate": float(y.mean()),
    "before_after": results.to_dict(orient="records"),
    "memorisation_gap": {k: float(v) for k, v in gap.items()},
}
with open("work/outputs/w06_validation_audit.json", "w") as f:
    json.dump(receipt, f, indent=2)
print("Wrote work/outputs/w06_validation_audit.json")

Wrote work/outputs/w06_validation_audit.json


**What the before/after shows.** The random split looks great — precision@20 of 0.90 — and the
grouped-client split drops it to about 0.45, close to *half*. That ~0.45 fall is the **memorisation
effect**: under a random split the model partly recognised *which client* a page came from rather
than reading the page's own signals. Both are far above the base rate (0.19), so the model still
carries real, generalisable signal — but the honest number is the grouped one, and the *gap* is the
thing I had not previously quantified.

Directional reading, not a promise: *on clients the model has never seen, the measured precision@20
is roughly half of what a random split would have advertised.* That is the number I will quote
going forward.

## 3. Leakage audit

The same hunt from Week 3, run on my **final** feature set. I attack my own model on three fronts:

1. **Label-derived features.** My label is `ctr_vs_tier < 0.50`, where `ctr = clicks / impressions`.
   So `clicks_90d` and `ctr` *define* the label. I exclude both. **Test:** deliberately add
   `clicks_90d` back and watch the score — if it does not jump toward 1.0, my harness is broken.
2. **Decision-derived / label-trap columns.** `trend_direction`, `trend_pct`, `is_declining_label`
   are label-family columns; `ctr_vs_tier` is the label source itself. All excluded.
3. **Grouping columns.** `client_id` and `content_id` are for *splitting and joining only* — never
   features. Verified by asserting they are absent from `X`.

In [5]:
# Leakage test 1: ADD the label-defining column back and confirm the score collapses toward 1.0.
X_leaky = pd.concat([X, vis[["clicks_90d"]].fillna(0)], axis=1)
def oof_predictions_leaky(F):
    preds = np.zeros(len(y))
    for tr, te in StratifiedKFold(n_splits=4, shuffle=True, random_state=SEED).split(F, y):
        m = RandomForestClassifier(n_estimators=300, max_depth=8, class_weight="balanced",
                                   random_state=SEED, n_jobs=-1)
        m.fit(F.iloc[tr], y[tr])
        preds[te] = m.predict_proba(F.iloc[te])[:, 1]
    return preds

leaky_preds = oof_predictions_leaky(X_leaky)
print("LEAKAGE PROBE — same random split, only difference is the feature set:")
print(f"  WITHOUT clicks_90d (honest) : P@50={precision_at_k(random_preds, y, 50):.3f} "
      f"| AP={average_precision_score(y, random_preds):.3f}")
print(f"  WITH    clicks_90d (leaky)  : P@50={precision_at_k(leaky_preds, y, 50):.3f} "
      f"| AP={average_precision_score(y, leaky_preds):.3f}")
print("  -> the jump toward 1.0 is the confession: clicks_90d reads the answer back.")

LEAKAGE PROBE — same random split, only difference is the feature set:
  WITHOUT clicks_90d (honest) : P@50=0.860 | AP=0.539
  WITH    clicks_90d (leaky)  : P@50=1.000 | AP=0.896
  -> the jump toward 1.0 is the confession: clicks_90d reads the answer back.


In [6]:
# Leakage test 2 & 3: prove the banned columns are NOT in the feature matrix.
banned = ["clicks_90d", "ctr", "ctr_vs_tier", "trend_direction", "trend_pct",
          "is_declining_label", "content_id", "client_id"]
feature_cols = list(X.columns)
raw_names = set(num_feats) | set(cat_feats)
present_banned = [c for c in banned if c in raw_names]
id_cols_in_X = [c for c in feature_cols if "client_id" in c or "content_id" in c]
print("Banned columns requested as features :", present_banned or "none -> clean")
print("ID columns leaked into X            :", id_cols_in_X or "none -> clean")

# Also confirm client_id is disjoint across the honest split (no group straddles train/test).
def disjoint_ok(splitter):
    folds = splitter.split(X, y, groups)
    for tr, te in folds:
        if set(groups[tr]) & set(groups[te]):
            return False
    return True
print("Grouped split leaves clients disjoint:", disjoint_ok(GroupKFold(n_splits=4)))

# Timeline check (statement, not a query): every feature is a 90d/cumulative snapshot known at the
# decision moment; the label is a within-snapshot ratio, so no feature window contains a FUTURE label.
print("Feature timeline: all features are observed snapshot fields (no future window) -> legal.")

Banned columns requested as features : none -> clean
ID columns leaked into X            : none -> clean
Grouped split leaves clients disjoint: True
Feature timeline: all features are observed snapshot fields (no future window) -> legal.


**Leakage audit result.** The honest feature set is clean: no label-defining column, no
label-trap column, no ID column, and the grouped split leaves clients fully disjoint across
train/test. The deliberate-leak probe jumps from P@50 ≈ 0.86 to **1.00** when `clicks_90d` is added —
confirming both that the leak is real and that my test harness detects it. The disclosure I owe:
my proxy label (`ctr_vs_tier`) is itself *derived from* CTR, so any feature strongly correlated with
CTR — most notably `impressions_90d` and `sessions_90d` — carries a **weaker but non-zero** version
of the same circularity. That is a limitation of the *proxy*, not a bug in the split, and it is why
my precision numbers should be read as *opportunity-ranking* support, not as validated truth.

## 4. Claim rewrite

Time to turn the lens on my own boldest sentence. In Week 5 I was tempted to write something like:

> ❌ *"My random forest proves which pages are under-capturing clicks and should be refreshed."*

That is three errors in one sentence: **proves** (causal overreach from an observational proxy),
**my random forest** (the strongest model, not the honest number), and **should be refreshed** (an
action claim the validation does not support).

**Rewritten in safe language:**

> ✅ *"On a client-holdout (grouped) validation, a random forest measured a precision@20 of about
> 0.45 — roughly double the 0.19 base rate — in ranking pages by a **proxy** for CTR
> under-capture. The ranking is **directional, decision-support**: it suggests which pages a
> reviewer might look at first, and does **not** establish that refreshing a page causes clicks to
> rise. The proxy label is derived from CTR itself, so part of the measured skill reflects that
> definition rather than independent truth."*

The rewrite keeps every number, drops every overreach, and adds the one disclosure that matters
(the proxy circularity). Same evidence, chosen door — the rule the worked-example papers are built
on.

In [7]:
# Final safe-claim summary: each claim paired with the exact evidence and its honest bound.
summary = pd.DataFrame([
    {"claim": "Honest (grouped-client) precision@20",
     "measured": round(float(results.loc[1, 'P@20']), 3),
     "base_rate": round(float(y.mean()), 3),
     "language": "measured, directional"},
    {"claim": "Memorisation gap (random - grouped) P@20",
     "measured": round(float(gap['P@20']), 3),
     "base_rate": None,
     "language": "observed"},
    {"claim": "Leaky variant P@50 (with clicks_90d)",
     "measured": round(float(precision_at_k(leaky_preds, y, 50)), 3),
     "base_rate": round(float(y.mean()), 3),
     "language": "demonstrated leak, discarded"},
    {"claim": "Effect of refreshing a page on clicks",
     "measured": "not measured",
     "base_rate": None,
     "language": "out of scope — no causal claim"},
])
print(summary.to_string(index=False))

                                   claim     measured  base_rate                       language
    Honest (grouped-client) precision@20         0.45      0.193          measured, directional
Memorisation gap (random - grouped) P@20         0.45        NaN                       observed
    Leaky variant P@50 (with clicks_90d)          1.0      0.193   demonstrated leak, discarded
   Effect of refreshing a page on clicks not measured        NaN out of scope — no causal claim


## Self-check

Before submitting, each line confirmed honestly:

- [x] **Every section filled** — markdown thinking *and* the code that backs each number
- [x] **Runs top to bottom, no errors** — executed with nbconvert, zero error outputs
- [x] **No client names, URLs, or private queries** — only the anonymised starter slice, aggregate numbers
- [x] **Careful words everywhere** — observed / measured / directional / decision-support; no causal verbs
- [x] **Committed under `work/notebooks/`** — `w06_validation_audit.ipynb`, metrics receipt in `work/outputs/`

**One-line takeaway:** the paper earned trust by *disclosing its design*; my job was to give my own
model the same treatment — an honest split, a leakage hunt, real failure examples, and a rewritten
claim that says no more than the evidence supports.